# Round 1 Data Analysis

Two products: **ASH_COATED_OSMIUM** and **INTARIAN_PEPPER_ROOT**. They have completely different data-generating processes and require different strategies.

**Calibration rule**: never examine a variable in isolation — always condition on every other known variable before declaring structure.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA = 'data/'
COLORS = {0: '#1f77b4', -1: '#ff7f0e', -2: '#2ca02c'}

price_files = [(0, 'prices_round_1_day_0.csv'), (-1, 'prices_round_1_day_-1.csv'), (-2, 'prices_round_1_day_-2.csv')]
trade_files = [(0, 'trades_round_1_day_0.csv'), (-1, 'trades_round_1_day_-1.csv'), (-2, 'trades_round_1_day_-2.csv')]

prices = pd.concat([pd.read_csv(DATA + f, sep=';').assign(day_label=d) for d, f in price_files], ignore_index=True)
trades = pd.concat([pd.read_csv(DATA + f, sep=';').assign(day_label=d) for d, f in trade_files], ignore_index=True)

ash = prices[prices['product'] == 'ASH_COATED_OSMIUM'].copy().sort_values(['day_label', 'timestamp'])
pep = prices[prices['product'] == 'INTARIAN_PEPPER_ROOT'].copy().sort_values(['day_label', 'timestamp'])
ash_trades = trades[trades['symbol'] == 'ASH_COATED_OSMIUM'].copy()
pep_trades  = trades[trades['symbol'] == 'INTARIAN_PEPPER_ROOT'].copy()

print(f'Price rows: {len(prices):,}  |  Trade rows: {len(trades):,}')
print(f'ASH: {len(ash):,} rows  |  Pepper: {len(pep):,} rows')

---
## 1. ASH_COATED_OSMIUM — Fair Value is a Constant

**Hypothesis**: FV = 10000, pinned by game design. The observed oscillations are entirely bid-ask bounce, not genuine price discovery.

~14 rows per day have `mid_price == 0` (empty book at startup). Excluded from all statistics.

In [ ]:
ash_valid = ash[ash['mid_price'] > 0].copy()

stats = ash_valid.groupby('day_label')['mid_price'].agg(['mean', 'std', 'min', 'max']).round(2)
stats.columns = ['mean', 'stdev', 'min', 'max']
print('ASH mid-price stats (excluding empty-book rows):')
print(stats)
print(f'\nCross-day mean range: {stats["mean"].max() - stats["mean"].min():.2f} ticks')
print(f'Max excursion from FV=10000: {(ash_valid["mid_price"] - 10000).abs().max():.0f} ticks')

In [ ]:
fig = go.Figure()
for day in [0, -1, -2]:
    d = ash_valid[ash_valid['day_label'] == day]
    fig.add_trace(go.Scatter(
        x=d['timestamp'], y=d['mid_price'],
        name=f'day {day}', line=dict(color=COLORS[day], width=1), opacity=0.8
    ))
fig.add_hline(y=10000, line_dash='dash', line_color='red', annotation_text='FV = 10000')
fig.update_layout(
    title='ASH Mid-Price: All Three Days',
    xaxis_title='Timestamp', yaxis_title='Mid Price', height=400
)
fig.show()

### 1a. The −0.5 Autocorrelation Test

Under Roll's model, lag-1 return autocorrelation = −s²/(2σ²_p + s²).
When this equals exactly **−0.5**, it means σ²_p ≈ 0: the underlying true price does not move at all. All observed price changes are pure bid-ask bounce around a fixed FV.

A genuine mean-reverting AR(1) process would produce negative autocorrelations decaying over multiple lags. What we expect for a game-constant FV: a single spike at lag 1 = −0.5 and nothing after.

In [ ]:
def lag_autocorr(series, max_lag=6):
    r = series.diff().dropna().values
    mean_r = r.mean()
    var_r = ((r - mean_r) ** 2).mean()
    if var_r == 0:
        return {k: np.nan for k in range(1, max_lag + 1)}
    return {
        lag: ((r[lag:] - mean_r) * (r[:-lag] - mean_r)).mean() / var_r
        for lag in range(1, max_lag + 1)
    }

pep_valid = pep[pep['mid_price'] > 0].copy()

for name, df in [('ASH', ash_valid), ('Pepper', pep_valid)]:
    mid = df.sort_values(['day_label', 'timestamp'])['mid_price'].reset_index(drop=True)
    ac = lag_autocorr(mid)
    print(f'{name} return autocorrelations:')
    for lag, val in ac.items():
        note = '  <-- exact bid-ask bounce signature' if abs(val + 0.5) < 0.01 else ''
        print(f'  lag={lag}: {val:+.4f}{note}')
    print()

**Result**: lag-1 = −0.5 exactly on both products, lags 2+ ≈ 0. This is the fingerprint of a fixed FV with all movement from spread crossing — not a stochastic process. The FV is not mean-reverting; it is literally stationary by game design.

---
## 2. INTARIAN_PEPPER_ROOT — Perfectly Linear Trend

**Hypothesis**: FV follows a deterministic ramp:

```
FV = 10000 + (day + 2) * 1000 + timestamp / 1000
```

This gives +1 tick per 1000 timestamps = **+1000 ticks per full day**.

In [ ]:
pep_valid['fv']       = 10000 + (pep_valid['day_label'] + 2) * 1000 + pep_valid['timestamp'] / 1000
pep_valid['residual'] = pep_valid['mid_price'] - pep_valid['fv']

resid = pep_valid.groupby('day_label')['residual'].agg(['mean', 'std', 'min', 'max']).round(3)
resid.columns = ['mean', 'stdev', 'min', 'max']
print('Pepper residuals (mid − predicted FV):')
print(resid)
print()
nonzero = pep_valid.groupby('day_label').apply(lambda g: (g['residual'].abs() > 0.4).mean())
print('Fraction of ticks that deviate at all from the formula:')
print(nonzero.round(3))

In [ ]:
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Pepper Mid-Price vs Predicted FV (dashed)',
    'Residuals (mid − FV)'
])
for day in [0, -1, -2]:
    d = pep_valid[pep_valid['day_label'] == day]
    fig.add_trace(go.Scatter(
        x=d['timestamp'], y=d['mid_price'],
        name=f'mid day {day}', line=dict(color=COLORS[day], width=1)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=d['timestamp'], y=d['fv'],
        name=f'FV day {day}', line=dict(color=COLORS[day], width=2, dash='dash'), showlegend=False
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=d['timestamp'], y=d['residual'],
        name=f'resid day {day}', line=dict(color=COLORS[day], width=1), showlegend=False
    ), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=2, col=1)
fig.update_layout(height=600, title='Pepper: Deterministic Linear FV')
fig.show()

The formula fits perfectly — residuals have mean ≈ 0 and range ±10 ticks across all three days. This is game-designed: same structure as ASH (game-constant FV) but with a non-zero trend slope. You can compute exact FV in the trader at every tick with no estimation.

---
## 3. Book Depth Structure

In [ ]:
depth_cols = ['bid_price_1','bid_price_2','bid_price_3','ask_price_1','ask_price_2','ask_price_3']
for prod_name, df in [('ASH', ash), ('Pepper', pep)]:
    d0 = df[df['day_label'] == 0]
    n = len(d0)
    print(f'{prod_name} depth presence (day 0, n={n:,}):')
    for col in depth_cols:
        pct = d0[col].notna().mean()
        print(f'  {col}: {pct:.1%}')
    print()

L1 ≈ 96%, L2 ≈ 65%, L3 ≈ 2%. L3 is noise. **~35% of all ticks are thin-book** (L2 absent while L1 is present) — stable across both products and all three days.

---
## 4. Spread–Thin-Book Structural Relationship (ASH)

**Thin-book** = `bid_price_2` is NaN while `bid_price_1` is present.

Following the calibration rule: never examine spread or thin-book in isolation. Condition each on the other.

In [ ]:
ash0 = ash[(ash['day_label'] == 0) & ash['bid_price_1'].notna() & ash['ask_price_1'].notna()].copy()
ash0['spread']    = ash0['ask_price_1'] - ash0['bid_price_1']
ash0['thin_bid']  = ash0['bid_price_2'].isna()
ash0['thin_ask']  = ash0['ask_price_2'].isna()
ash0['thin_both'] = ash0['thin_bid'] & ash0['thin_ask']

print('Thin-book frequency (ASH day 0):')
for col in ['thin_bid', 'thin_ask', 'thin_both']:
    print(f'  {col}: {ash0[col].mean():.1%}')
print()

spread_tbl = (ash0.groupby('spread')
                  .agg(count=('spread', 'size'), thin_bid_pct=('thin_bid', 'mean'))
                  .reset_index())
spread_tbl = spread_tbl[spread_tbl['count'] >= 10].copy()
spread_tbl['pct_ticks']    = (spread_tbl['count'] / len(ash0) * 100).round(1)
spread_tbl['thin_bid_pct'] = (spread_tbl['thin_bid_pct'] * 100).round(1)
print('Spread | Count | % ticks | Thin-bid %')
print(spread_tbl[['spread','count','pct_ticks','thin_bid_pct']].to_string(index=False))

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Spread distribution (% of ticks)',
    'Thin-bid rate conditional on spread'
])
fig.add_trace(go.Bar(
    x=spread_tbl['spread'], y=spread_tbl['pct_ticks'], marker_color='steelblue'
), row=1, col=1)
fig.add_trace(go.Bar(
    x=spread_tbl['spread'], y=spread_tbl['thin_bid_pct'], marker_color='orangered'
), row=1, col=2)
fig.update_xaxes(title_text='Spread (ticks)')
fig.update_yaxes(title_text='% of ticks', row=1, col=1)
fig.update_yaxes(title_text='Thin-bid rate (%)', row=1, col=2)
fig.update_layout(
    height=400, title='ASH: Spread ↔ Thin-Book Structure (Day 0)', showlegend=False
)
fig.show()

**Key finding**: Thin-bid rate climbs from 21% at spread=16 to **100%** at spread=21. These are not independent signals.

Causal chain: **price deviates far from FV → market maker stress → L2 withdrawn → spread widens → thin + wide spread jointly observed**.

Thin-book is the observable output of market maker stress, not an exogenous shock to watch for.

---
## 5. L1 Volume Bimodality — Explained by Conditioning

The marginal `bid_volume_1` distribution looks bimodal. Conditioning on L2 presence resolves it into two completely separate bot behaviors.

In [ ]:
ash0_bvol = ash0.dropna(subset=['bid_price_1', 'bid_volume_1']).copy()
normal_vols = ash0_bvol[~ash0_bvol['thin_bid']]['bid_volume_1']
thin_vols   = ash0_bvol[ ash0_bvol['thin_bid']]['bid_volume_1']

print(f'Normal (has L2): n={len(normal_vols):,}, mean={normal_vols.mean():.1f}, '
      f'range=[{normal_vols.min():.0f}, {normal_vols.max():.0f}]')
print(f'Thin (no L2):    n={len(thin_vols):,}, mean={thin_vols.mean():.1f}, '
      f'range=[{thin_vols.min():.0f}, {thin_vols.max():.0f}]')

l2_vols = ash0.dropna(subset=['bid_volume_2'])['bid_volume_2']
print(f'\nL2 volume when present: n={len(l2_vols):,}, mean={l2_vols.mean():.1f}, '
      f'range=[{l2_vols.min():.0f}, {l2_vols.max():.0f}]')
print('L2 vol counts:', dict(sorted(l2_vols.value_counts().items())))

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'bid_vol_1 — marginal (looks bimodal)',
    'bid_vol_1 | normal book (L2 present)',
    'bid_vol_1 | thin book (L2 absent)'
])
for col, vols, color in [
    (1, ash0_bvol['bid_volume_1'], 'gray'),
    (2, normal_vols, 'steelblue'),
    (3, thin_vols,   'orangered')
]:
    vc = vols.value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index, y=vc.values, marker_color=color), row=1, col=col)
fig.update_xaxes(title_text='bid_volume_1')
fig.update_layout(
    height=380,
    title='L1 Volume Bimodality Explained: Condition on L2 Presence (ASH Day 0)',
    showlegend=False
)
fig.show()

**Key finding**:
- **Normal book (L2 present)**: L1 volume concentrated at [10–15]. Shallow L1 + deep L2 buffer.
- **Thin book (no L2)**: bimodal — either [10–15] (residual normal quote) OR [24–30] (large defensive block).

The [24–30] block **only appears when L2 is absent**. This is bot decision logic made visible: when stressed enough to pull L2, the market maker sometimes switches to a large protective block at L1 instead.

L2 volume when present is uniform [20–30], mean ≈ 24 — a flat block, not a gradient. The market maker treats L2 as binary: fully deployed or fully withdrawn.

---
## 6. Trade Execution — All Trades Cross L1

In [ ]:
book_d0 = ash[(ash['day_label'] == 0) & ash['bid_price_1'].notna() & ash['ask_price_1'].notna()].copy()
book_d0['spread'] = book_d0['ask_price_1'] - book_d0['bid_price_1']
book_d0 = book_d0[['timestamp','bid_price_1','ask_price_1','spread']].rename(
    columns={'bid_price_1': 'book_bid1', 'ask_price_1': 'book_ask1'})

atr0 = ash_trades[ash_trades['day_label'] == 0].merge(book_d0, on='timestamp', how='left')
atr0['crossed'] = np.where(
    (atr0['price'] - atr0['book_ask1']).abs() < 0.01, 'hit_ask',
    np.where((atr0['price'] - atr0['book_bid1']).abs() < 0.01, 'hit_bid', 'other')
)

print('Trade classification (ASH day 0):')
print(atr0['crossed'].value_counts())
print()

sp_counts = atr0.groupby(atr0['spread'].round(0))['price'].count()
total = len(atr0)
print('Spread active at trade time:')
for sp, cnt in sorted(sp_counts.items()):
    if cnt > 0:
        print(f'  spread={sp:.0f}: {cnt} trades ({100*cnt/total:.1f}%)')

**Key finding**: 100% of trades execute at exactly `bid_price_1` or `ask_price_1` — zero midpoint fills. You must always cross the spread. ~60% of trades occur at spread=16 (dominant regime). Spread=18/19/21 trades are slightly over-represented relative to their tick share (~32% of trades vs ~24% of ticks), suggesting more urgency when the book is stressed.

---
## 7. Market Maker Quote Distribution (ASH)

In [ ]:
ash0v = ash0.dropna(subset=['bid_price_1', 'ask_price_1']).copy()
bid_dist = (ash0v['bid_price_1'] - 10000).value_counts().sort_index()
ask_dist = (ash0v['ask_price_1'] - 10000).value_counts().sort_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Bid price offset from FV=10000',
    'Ask price offset from FV=10000'
])
fig.add_trace(go.Bar(x=bid_dist.index, y=bid_dist.values, marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Bar(x=ask_dist.index, y=ask_dist.values, marker_color='orangered'), row=1, col=2)
for col in [1, 2]:
    fig.add_vline(x=0, line_dash='dash', line_color='black', row=1, col=col)
fig.update_xaxes(title_text='Distance from FV (ticks)')
fig.update_layout(height=400, title='ASH: Market Maker Quote Distribution (Day 0)', showlegend=False)
fig.show()

print(f'Bid peak: FV {bid_dist.idxmax():+.0f} ticks')
print(f'Ask peak: FV {ask_dist.idxmax():+.0f} ticks')
print(f'Spread at peaks: {ask_dist.idxmax() - bid_dist.idxmax():.0f} ticks')

---
## 8. Cross-Day Consistency

Per the calibration rule: every finding must hold on 2+ independent days.

In [ ]:
rows = []
for day in [0, -1, -2]:
    d = ash[(ash['day_label'] == day) & ash['bid_price_1'].notna() & ash['ask_price_1'].notna()].copy()
    d['spread']   = d['ask_price_1'] - d['bid_price_1']
    d['thin_bid'] = d['bid_price_2'].isna()
    mid_v = d[d['mid_price'] > 0]['mid_price']
    lag1  = mid_v.diff().dropna().autocorr(lag=1)
    rows.append({
        'day': day,
        'mid_mean':      round(mid_v.mean(), 2),
        'mid_stdev':     round(mid_v.std(), 2),
        'dom_spread':    d['spread'].mode()[0],
        'thin_bid_%':    f"{d['thin_bid'].mean():.1%}",
        'thin|sp=16_%':  f"{d[d['spread']==16]['thin_bid'].mean():.1%}" if (d['spread']==16).any() else 'n/a',
        'thin|sp=21_%':  f"{d[d['spread']==21]['thin_bid'].mean():.1%}" if (d['spread']==21).any() else 'n/a',
        'lag1_ac':       f'{lag1:.4f}'
    })

print(pd.DataFrame(rows).set_index('day').to_string())

---
## 9. Tick Size

In [ ]:
price_cols = ['bid_price_1','bid_price_2','bid_price_3','ask_price_1','ask_price_2','ask_price_3']
all_prices = pd.concat([ash[c].dropna() for c in price_cols]).sort_values().unique()
diffs = np.diff(all_prices)
diffs = diffs[diffs > 0]
print(f'ASH tick increments: {np.unique(diffs.round(1))}  (all exactly 1 tick)')
print(f'ASH price range: [{all_prices.min():.0f}, {all_prices.max():.0f}]  ({all_prices.max() - all_prices.min():.0f} ticks wide)')

---
## 10. Summary — Strategy Implications

### ASH_COATED_OSMIUM — Market Making

| Finding | Implication |
|---|---|
| FV = 10000 is a game constant | Hardcode `fair_value = 10000`. Never estimate it from a rolling mean. |
| Lag-1 autocorr = −0.5 exactly | All price movement is bid-ask bounce. No underlying drift. |
| Dominant spread = 16 ticks | Market maker quotes ≈ FV±8. Quote FV±7 to undercut and attract flow. |
| Wider spread is 100% explained by thin book | Thin book = market maker stress state. Spread=21 + thin = maximally stressed. |
| Large L1 vol [24–30] only when thin | Defensive block. Market maker has pulled L2 and posted a large guard. |
| All trades at bid_price_1 or ask_price_1 | Must cross the spread. Factor half-spread into every take decision. |

```python
# In trader:
fair_value = 10000
# Quote at fair_value - 7 (bid) and fair_value + 7 (ask)
# Thin-book detection:
# if len(state.order_depths['ASH_COATED_OSMIUM'].buy_orders) == 1:
#     price is dislocated from FV — consider taking inventory
```

### INTARIAN_PEPPER_ROOT — Trend Following

| Finding | Implication |
|---|---|
| `FV = 10000 + (day+2)*1000 + ts/1000` | Compute exact FV every tick. No estimation needed. |
| Residuals ±10 ticks around FV | Small noise band. Dominant spread 13–14. Cross when mid deviates meaningfully. |
| FV increases +1 tick per 1000 timestamps | Always net long. Open long early in the day and ride the trend. |

```python
# In trader:
fair_value = 10000 + (day + 2) * 1000 + state.timestamp / 1000
# Buy when best_ask < fair_value - threshold
# Sell when best_bid > fair_value + threshold
```